# 03 — Candidate Generation: Repurchase + Trending

**Purpose:** two cheap, high-recall candidate sources (Stage-1 'Net').

We generate candidates for **both** label weeks, each using only data *before* that
week's start (no leakage):
- **train** week -> for users who bought in the TRAIN week, basis `t_dat < train_week_start`
- **val** week   -> for users who bought in the VAL week,   basis `t_dat < val_week_start`

Trending is a small global list (expanded to users later in notebook 05) to stay memory-light.

In [1]:
from recsys_utils import *
import numpy as np, pandas as pd
mlflow = setup_mlflow()

transactions = load_parquet(PROCESSED_DIR / 'transactions_clean.parquet')
cfg          = load_json(PROCESSED_DIR / 'split_config.json')
train_truth  = load_parquet(PROCESSED_DIR / 'train_truth.parquet')
val_truth    = load_parquet(PROCESSED_DIR / 'val_truth.parquet')

In [2]:
REPURCHASE_N = 20    # last N distinct articles a user bought
TREND_DAYS   = 14    # trending window
TREND_K      = 100   # size of the global trending list we keep

In [3]:
def repurchase_candidates(tx, basis_end, users):
    users = np.asarray(list(users))
    hist = tx[(tx['t_dat'] < basis_end) & (tx['customer_id'].isin(users))]
    hist = hist.sort_values(['customer_id', 't_dat'])
    rep  = (hist.groupby('customer_id', group_keys=False)
                .tail(REPURCHASE_N)[['customer_id', 'article_id']]
                .drop_duplicates())
    rep['source'] = 'repurchase'
    return rep.reset_index(drop=True)

def trending_candidates(tx, basis_end, k=TREND_K, days=TREND_DAYS):
    win = tx[(tx['t_dat'] < basis_end) & (tx['t_dat'] >= basis_end - pd.Timedelta(days=days))]
    t = (win['article_id'].value_counts().head(k)
            .rename_axis('article_id').reset_index(name='recent_count'))
    t['trending_rank'] = np.arange(1, len(t) + 1, dtype='int16')
    t['source'] = 'trending'
    return t

In [4]:
weeks = [
    ('train', train_truth['customer_id'].values, pd.Timestamp(cfg['train_week_start'])),
    ('val',   val_truth['customer_id'].values,   pd.Timestamp(cfg['val_week_start'])),
]

for tag, users, basis in weeks:
    rep = repurchase_candidates(transactions, basis, users)
    trd = trending_candidates(transactions, basis)
    save_parquet(rep, CANDIDATE_DIR / f'repurchase_{tag}.parquet')
    save_parquet(trd, CANDIDATE_DIR / f'trending_{tag}.parquet')
    print(f'[{tag}] repurchase rows={len(rep):,}  users={rep.customer_id.nunique():,}  '
          f'trending items={len(trd)}')

Saved: C:\Users\Michael Fulling\H&M Project\data\processed\candidates\repurchase_train.parquet  shape=(1014895, 3)
Saved: C:\Users\Michael Fulling\H&M Project\data\processed\candidates\trending_train.parquet  shape=(100, 4)
[train] repurchase rows=1,014,895  users=66,624  trending items=100
Saved: C:\Users\Michael Fulling\H&M Project\data\processed\candidates\repurchase_val.parquet  shape=(966427, 3)
Saved: C:\Users\Michael Fulling\H&M Project\data\processed\candidates\trending_val.parquet  shape=(100, 4)
[val] repurchase rows=966,427  users=63,412  trending items=100


In [5]:
with mlflow.start_run(run_name='03_candidates_repurchase_trending'):
    mlflow.log_param('repurchase_n', REPURCHASE_N)
    mlflow.log_param('trend_days', TREND_DAYS)
    mlflow.log_param('trend_k', TREND_K)
print('Logged candidate-generation run.')

2026/06/11 12:09:30 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



Logged candidate-generation run.
